<a href="https://colab.research.google.com/github/yaaz15/yazdi-ahmadinejad-/blob/main/regresi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import plotly.graph_objects as go
import plotly.io as pio

# Memaksa Plotly untuk merender grafik interaktif langsung di dalam sel Google Colab
pio.renderers.default = 'colab'

# 1. MEMBACA DATA ASLI DARI GITHUB (KOTA BOGOR)

url = 'https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv'
df_all = pd.read_csv(url)

# Menentukan kolom wilayah dan menyaring khusus Kota Bogor
kolom_kota = 'Kota' if 'Kota' in df_all.columns else [c for c in df_all.columns if 'kota' in c.lower() or 'kab' in c.lower()][0]

# Memfilter Kota Bogor (serta memastikan Kabupaten Bogor tidak ikut terambil)
df = df_all[df_all[kolom_kota].astype(str).str.contains('Bogor', case=False, na=False)].copy()
if df[kolom_kota].str.contains('Kabupaten', case=False).any():
    df = df[df[kolom_kota].astype(str).str.contains('Kota Bogor', case=False, na=False)].copy()

# Menampilkan data Kota Bogor
print("Data Kota Bogor:")
display(df)
print(f"Total data Kota Bogor yang dianalisis: {len(df)} baris.\n")

Data Kota Bogor:


,Kota,Tahun,DBH_PAJAK,DBH_SDA,DBH_Lainnya,DAU_Grant,DAU_Earmark,DAK_Fisik,DAK_Nonfisik,PAD,...,bm_lingkungan,bm_ekonomi,bm_infrastruktur,bm_pariwisata,bm_perlindungan_sosial,KEMISKINAN,PENGANGGURAN,KETIMPANGAN,IPM,PE
374,Kota Bogor,2014,39.11,50.63,18.05,89.14,40.31,46.93,69.12,83.38,...,142.87,264.63,276.75,234.20,166.17,19.49,4.44,0.45,73.67,6.39
375,Kota Bogor,2015,59.84,39.84,23.02,83.42,73.80,51.13,52.84,135.54,...,98.79,82.92,92.19,163.35,255.18,10.08,9.66,0.45,74.94,6.66
376,Kota Bogor,2016,43.16,31.13,30.12,91.56,71.70,78.91,58.54,110.36,...,172.35,226.26,292.08,297.32,203.64,19.42,6.11,0.40,68.79,7.23
377,Kota Bogor,2017,68.90,55.43,39.78,102.84,62.96,81.18,59.87,157.56,...,235.41,235.49,163.22,63.69,258.04,19.11,7.36,0.45,80.13,3.29
378,Kota Bogor,2018,64.22,25.38,32.27,123.09,66.77,80.51,62.47,150.51,...,173.14,219.43,286.43,138.75,173.50,7.80,8.25,0.41,65.10,2.50
379,Kota Bogor,2019,74.52,41.88,25.91,126.46,73.10,83.41,60.82,125.73,...,170.80,113.08,219.21,170.48,79.62,18.47,6.68,0.44,66.30,4.90
380,Kota Bogor,2020,51.64,32.45,23.00,93.73,90.88,57.10,76.45,169.07,...,259.56,268.26,125.40,175.78,53.41,13.26,4.07,0.45,75.01,5.08
381,Kota Bogor,2021,91.91,76.62,40.86,121.61,85.61,55.49,98.54,116.26,...,140.35,92.21,97.08,128.98,125.80,9.92,8.71,0.42,69.84,6.98
382,Kota Bogor,2022,92.33,41.82,46.54,103.95,88.46,91.98,58.84,142.42,...,264.95,63.41,125.93,232.31,224.50,7.48,7.95,0.41,75.88,4.66
383,Kota Bogor,2023,55.93,24.99,30.51,118.14,85.75,76.93,66.25,180.29,...,151.70,229.37,73.25,268.69,247.56,15.79,9.51,0.41,66.47,5.43


Total data Kota Bogor yang dianalisis: 11 baris.



In [2]:
# 2. MODEL REGRESI & UJI ASUMSI KLASIK (KOTA BOGOR)

# Pastikan variabel independen (X) dan dependen (Y) sesuai dengan nama kolom pada dataset Kota Bogor
X = df[['bm_pendidikan', 'bm_perlindungan_sosial']]
Y = df['KEMISKINAN']

X_sm = sm.add_constant(X)
model = sm.OLS(Y, X_sm).fit()

# Menampilkan Summary OLS
print(model.summary())

print("\n" + "="*50)
print("FUNGSI REGRESI (Kota Bogor):")
# Menyesuaikan penanganan tanda plus/minus pada koefisien agar rapi
coef_pendidikan = model.params['bm_pendidikan']
coef_sosial = model.params['bm_perlindungan_sosial']
sign_sosial = "+" if coef_sosial >= 0 else "-"

print(f"Y = {model.params['const']:.4f} + {coef_pendidikan:.4f}(BM_Pendidikan) {sign_sosial} {abs(coef_sosial):.4f}(BM_Sosial)")
print("="*50)

print("\nHASIL UJI ASUMSI KLASIK (KOTA BOGOR)")
# 1. Normalitas
shapiro_stat, shapiro_p = stats.shapiro(model.resid)
print(f"- Normalitas (Shapiro-Wilk) P-Value: {shapiro_p:.4f} (>0.05 = Lolos)")

# 2. Multikolinearitas (VIF)
vif_data = pd.DataFrame()
vif_data["Variabel"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(f"- Multikolinearitas (VIF)            : {vif_data['VIF'].max():.4f} (<10 = Lolos)")

# 3. Heteroskedastisitas
bp_test = sms.het_breuschpagan(model.resid, model.model.exog)
print(f"- Heteroskedastisitas (BP) P-Value   : {bp_test[1]:.4f} (>0.05 = Lolos)")

# 4. Autokorelasi
dw_value = sm.stats.stattools.durbin_watson(model.resid)
print(f"- Autokorelasi (Durbin-Watson)       : {dw_value:.4f} (Mendekati 2 = Lolos)\n")

                            OLS Regression Results                            
Dep. Variable:             KEMISKINAN   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                 -0.081
Method:                 Least Squares   F-statistic:                    0.6244
Date:                Thu, 15 Oct 2026   Prob (F-statistic):              0.560
Time:                        00:07:54   Log-Likelihood:                -31.344
No. Observations:                  11   AIC:                             68.69
Df Residuals:                       8   BIC:                             69.88
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     17

In [3]:
# 3. VISUALISASI 3D INTERAKTIF (KOTA BOGOR)

# Membuat grid bidang regresi berdasarkan rentang data Kota Bogor
x_range = np.linspace(df['bm_pendidikan'].min(), df['bm_pendidikan'].max(), 20)
y_range = np.linspace(df['bm_perlindungan_sosial'].min(), df['bm_perlindungan_sosial'].max(), 20)
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Menghitung prediksi Z untuk permukaan bidang regresi
z_grid = model.params['const'] + (model.params['bm_pendidikan'] * x_grid) + (model.params['bm_perlindungan_sosial'] * y_grid)

# Inisialisasi Plotly Figure
fig = go.Figure()

# Menambahkan Bidang Regresi Permukaan
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    name='Bidang Regresi',
    colorscale='viridis',
    opacity=0.8,
    showscale=False
))

# Menambahkan Titik Data Aktual Kota Bogor
fig.add_trace(go.Scatter3d(
    x=df['bm_pendidikan'],
    y=df['bm_perlindungan_sosial'],
    z=df['KEMISKINAN'],
    mode='markers',
    name='Data Aktual',
    marker=dict(size=6, color='red', line=dict(color='white', width=1))
))

# Mengatur Tampilan Layout Grafik
fig.update_layout(
    title=f"Plot Bidang Regresi 3D: Kemiskinan terhadap BM Pendidikan & BM Sosial (Kota Bogor) | R² = {model.rsquared:.2f}",
    template='plotly_dark',
    scene=dict(
        xaxis_title='BM Pendidikan',
        yaxis_title='BM Perlindungan Sosial',
        zaxis_title='Kemiskinan'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Menampilkan kanvas interaktif di bawah sel Colab
fig.show()